# Solutions - Datetime Functions Basics (Easy 12)


In [ ]:
from pyspark.sql import functions as F, types as T

trips = spark.table("samples.nyctaxi.trips")
transactions = spark.table("samples.bakehouse.sales_transactions")
bookings = spark.table("samples.wanderbricks.bookings")


## Problem 1 - Date Components

Extract year, month, day, hour, and minute from pickup datetime.


In [ ]:
result_1 = trips.select(
    "tpep_pickup_datetime",
    F.year("tpep_pickup_datetime").alias("year"),
    F.month("tpep_pickup_datetime").alias("month"),
    F.dayofmonth("tpep_pickup_datetime").alias("day"),
    F.hour("tpep_pickup_datetime").alias("hour"),
    F.minute("tpep_pickup_datetime").alias("minute")
)
result_1.display()


## Problem 2 - Trip Duration

Calculate trip duration in minutes and filter for positive durations.


In [ ]:
result_2 = trips.select(
    "tpep_pickup_datetime", "tpep_dropoff_datetime",
    F.round((F.col("tpep_dropoff_datetime").cast("long") - F.col("tpep_pickup_datetime").cast("long")) / 60, 1).alias("duration_minutes"),
    "trip_distance", "fare_amount"
).filter(F.col("duration_minutes") > 0)
result_2.display()


## Problem 3 - Day of Week Sales

Aggregate transaction counts and revenue by day of week.


In [ ]:
result_3 = transactions.withColumn(
    "dow", F.dayofweek("dateTime")
).withColumn("day_name",
    F.when(F.col("dow") == 1, "Sunday")
     .when(F.col("dow") == 2, "Monday")
     .when(F.col("dow") == 3, "Tuesday")
     .when(F.col("dow") == 4, "Wednesday")
     .when(F.col("dow") == 5, "Thursday")
     .when(F.col("dow") == 6, "Friday")
     .otherwise("Saturday")
).groupBy("day_name").agg(
    F.count("*").alias("transaction_count"),
    F.round(F.sum("totalPrice"), 2).alias("total_revenue")
)
result_3.display()


## Problem 4 - Stay Duration Stats

Calculate average, min, and max stay nights grouped by booking status.


In [ ]:
result_4 = bookings.withColumn(
    "nights", F.datediff("check_out", "check_in")
).groupBy("status").agg(
    F.round(F.avg("nights"), 1).alias("avg_nights"),
    F.min("nights").alias("min_nights"),
    F.max("nights").alias("max_nights"),
    F.count("*").alias("booking_count")
)
result_4.display()


## Problem 5 - Monthly Transactions

Aggregate transactions by month with a formatted month label.


In [ ]:
result_5 = transactions.withColumn(
    "month_label", F.date_format(F.date_trunc("month", F.col("dateTime")), "yyyy-MM")
).groupBy("month_label").agg(
    F.count("*").alias("transaction_count"),
    F.round(F.sum("totalPrice"), 2).alias("total_revenue")
)
result_5.display()
